In [ ]:
import gymnasium as gym
import math
import numpy as np

In [ ]:
#env = gym.make('FrozenLake-v1', desc=None, map_name="4x4", is_slippery=True)
env = gym.make('FrozenLake-v1', desc=None, map_name="8x8", is_slippery=True)

In [ ]:
env.reset()

In [ ]:
# Sample an episode from the MDP
def sample():
    obs, info = env.reset()
    episode = []
    accReward = 0
    while True:
        # We sample from each action uniformly, this is our policy for evaluating the
        # value function (ie the difference between evaluation (this) and control is
        # whether we are improving this policy or not
        action = env.action_space.sample()
        nextObs, reward, terminated, trunc, info = env.step(action)
        accReward += reward
        episode.append((obs, action, reward, nextObs, accReward))
        obs = nextObs
        if trunc or terminated:
            break
    return episode

In [ ]:
iter = 100
for i in range(iter):
    episode = sample()
    if episode[-1][-1] >= 1:
        print(episode)

In [ ]:
env.reset()

values = np.zeros((env.observation_space.n))
visit_count = np.zeros((env.observation_space.n))

lamb = 0.9 # Discount factor lambda

# Run Monte Carlo sampling for some number of iterations
iter = 100000
for i in range(iter):
    G_t_1 = 0 # Intialise return from after the final step to 0

    # Sample the episode    
    episode = sample()

    # Go through each step in the episode in reverse order
    for step in reversed(episode):
        
        o, a, r, n_o, accR = step
        
        # G_t = r_t + lamda * G_t+1
        G_t = r + lamb * G_t_1

        # Update the values with the value of the return from this state
        values[n_o] += G_t
        # Update the visit count so we can average to get the state values
        visit_count[n_o] += 1

        # Update G_(t+1) for the next iteration
        G_t_1 = G_t

In [ ]:
env.observation_space.n
int(math.sqrt(env.observation_space.n))

In [ ]:
width = int(math.sqrt(env.observation_space.n))

In [ ]:
def print_board(board):
    step = int(math.sqrt(env.observation_space.n))
    for i in np.arange(int(env.observation_space.n), step=step):
        for j in np.arange(step):
            print(f"{board[i+j]:.5f}", end=', ')
        print()

In [ ]:
# Print the accumulated value of each state (we need to average of the number of visits for the true MC estimates)
print_board(values)

In [ ]:
print_board(visit_count)

In [ ]:
# Print the MC value function estimates
print_board(values/visit_count)